In [14]:
import streamlit as st
from google import genai
from google.genai import types
import os
import json
from dotenv import load_dotenv
import ocr
import evaluation
load_dotenv()

True

In [15]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [16]:
with open("dbmsas2.pdf", "rb") as f:
    answer_sheet_bytes = f.read()

In [17]:
with open("dbmsak2.pdf", "rb") as f:
    answer_key_bytes = f.read()

In [18]:
contents = [
    types.Part.from_bytes(data=answer_key_bytes, mime_type='application/pdf')
]

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    config=types.GenerateContentConfig(system_instruction=ocr.build_ocr_prompt_for_answer_key()),
    contents=contents
)

answer_key_json = json.loads(response.text.strip())


In [19]:
answer_key_json

{'subject': 'database management system',
 'total_marks': 60,
 'questions': [{'question_number': '3.1',
   'question': 'Is $A \\rightarrow C$ direct or derived?',
   'expected_answer': "In the relation R(A, B, C) with dependencies {$A \\rightarrow B, B \\rightarrow C$}, the dependency $A \\rightarrow C$ is derived through transitivity. According to Armstrong's Axioms (Transitive Rule), if $A \\rightarrow B$ and $B \\rightarrow C$ hold, then $A \\rightarrow C$ must also hold.",
   'max_marks': 2,
   'any_one_part': False},
  {'question_number': '3.2',
   'question': 'Prime vs. Non-prime Attribute',
   'expected_answer': 'Prime Attribute: An attribute that is a member of any candidate key of the relation. Example: In a table Student(ID, Name) where ID is the primary key, ID is a prime attribute. Non-prime Attribute: An attribute that is not part of any candidate key. Example: In the same table, Name is a non-prime attribute.',
   'max_marks': 2,
   'any_one_part': False},
  {'question_nu

In [20]:
contents = [
    types.Part.from_bytes(data=answer_sheet_bytes, mime_type='application/pdf'),
]

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    config=types.GenerateContentConfig(system_instruction=ocr.build_ocr_prompt_for_answer_sheet(answer_key_json)),
    contents=contents
)

answer_sheet_json = json.loads(response.text.strip())

In [21]:
answer_sheet_json

{'answers': [{'question_number': '3.1',
   'student_answer': 'Given R (A,B,C) FD1: A -> B FD2: B -> C we get A -> B from fd1 by transitivity of fd2: B -> C we get A -> C So A -> C is a derived functional dependency through transitivity'},
  {'question_number': '3.2',
   'student_answer': 'Prime attribute, Non Prime attribute: * Prime attribute is a set of attribute which is used to uniquely identify every tuple in Relation Eg: consider relation Sno. studentname, dept_no with studentno we can get every record uniquely from relation * Non prime attribute is set of attributes which are not prime means with that attributes we cannot uniquely identify every tuple in Relation Eg: consider relation Sno. studentname, dept_no with studentname or dept_no we cannot get every record uniquely from relation'},
  {'question_number': '3.3',
   'student_answer': 'Given A -> B, A -> C, AB -> D A+ -> A, B, C, D closure of set {A} is {A, B, C, D}'},
  {'question_number': '3.4',
   'student_answer': 'Lossl

In [22]:
print(evaluation.build_evaluation_prompt(answer_key_json,answer_sheet_json))

You are an expert examiner for database management system.

Evaluate each student answer against the expected answer and assign marks.

Q3.1 [2 marks]: Is $A \rightarrow C$ direct or derived?
  Expected: In the relation R(A, B, C) with dependencies {$A \rightarrow B, B \rightarrow C$}, the dependency $A \rightarrow C$ is derived through transitivity. According to Armstrong's Axioms (Transitive Rule), if $A \rightarrow B$ and $B \rightarrow C$ hold, then $A \rightarrow C$ must also hold.
  Student wrote: Given R (A,B,C) FD1: A -> B FD2: B -> C we get A -> B from fd1 by transitivity of fd2: B -> C we get A -> C So A -> C is a derived functional dependency through transitivity
  Is it either or question :False

Q3.2 [2 marks]: Prime vs. Non-prime Attribute
  Expected: Prime Attribute: An attribute that is a member of any candidate key of the relation. Example: In a table Student(ID, Name) where ID is the primary key, ID is a prime attribute. Non-prime Attribute: An attribute that is not p

In [23]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    config=types.GenerateContentConfig(system_instruction="You are an expert examiner"),
    contents=evaluation.build_evaluation_prompt(answer_key_json,answer_sheet_json)
)

evaluation_json = json.loads(response.text.strip())

In [24]:
evaluation_json

{'overall_feedback': "The student demonstrates a solid understanding of database normalization concepts, including Armstrong's Axioms and the progression of normal forms. While there are minor terminological inaccuracies and a missed partial dependency in the normalization task, the conceptual application is generally strong.",
 'evaluations': [{'question_number': '3.1',
   'student_answer': 'Given R (A,B,C) FD1: A -> B FD2: B -> C we get A -> B from fd1 by transitivity of fd2: B -> C we get A -> C So A -> C is a derived functional dependency through transitivity',
   'marks_awarded': 2,
   'feedback': 'Correct. The student accurately identified the dependency as derived and correctly cited transitivity as the reason.'},
  {'question_number': '3.2',
   'student_answer': 'Prime attribute, Non Prime attribute: * Prime attribute is a set of attribute which is used to uniquely identify every tuple in Relation Eg: consider relation Sno. studentname, dept_no with studentno we can get every r

In [25]:
score = sum([e['marks_awarded']  for e in evaluation_json['evaluations']])  # Placeholder value
description = evaluation_json['overall_feedback']
print(score,description)
        

25.0 The student demonstrates a solid understanding of database normalization concepts, including Armstrong's Axioms and the progression of normal forms. While there are minor terminological inaccuracies and a missed partial dependency in the normalization task, the conceptual application is generally strong.
